# EGC5310 — Semana 06 · Notebook Professor

## Uso em sala

Este notebook acompanha **a mesma ordem do Notebook Estudante**. Use-o para resolver as atividades junto com a turma.

As soluções aparecem logo após a tarefa correspondente. Não é apenas um gabarito: cada bloco destaca **o que perguntar, o que observar, erros comuns e a conexão conceitual**.

Fluxo recomendado:

**Slides/Mestre → Estudante tenta → Professor resolve/discute → retorno aos Slides.**


## Preparação — o dataset real

**Pergunta aos estudantes:** o que significa uma linha deste dataset? `StockCode` é necessariamente único?

**O que observar:** uma linha é item de transação. A visão `código → descrição` usada nas buscas terá cardinalidade muito menor que as ~541 mil linhas transacionais.


In [ ]:
import io, zipfile, urllib.request, pathlib
import pandas as pd
import numpy as np

URL = "https://archive.ics.uci.edu/static/public/352/online%2Bretail.zip"
ARQUIVO = pathlib.Path("Online Retail.xlsx")

if not ARQUIVO.exists():
    try:
        with urllib.request.urlopen(URL, timeout=90) as resposta:
            pacote = resposta.read()
        with zipfile.ZipFile(io.BytesIO(pacote)) as zipado:
            nome = next(nome for nome in zipado.namelist() if nome.lower().endswith(".xlsx"))
            ARQUIVO.write_bytes(zipado.read(nome))
    except Exception as erro:
        print("Download indisponível:", erro)
        print("Baixe o arquivo na página da UCI e envie Online Retail.xlsx para a sessão.")
        if 'google.colab' in __import__('sys').modules:
            from google.colab import files
            files.upload()
        if not ARQUIVO.exists():
            raise RuntimeError("Envie Online Retail.xlsx e execute novamente esta célula") from erro

# O Excel original tem aproximadamente 541 mil linhas.
df = pd.read_excel(ARQUIVO, engine="openpyxl")
print("Linhas:", len(df))
print("Colunas:", list(df.columns))
display(df.head(4))


In [ ]:
# Uma linha do dataset representa um item de uma fatura.
# O mesmo StockCode pode aparecer em muitas transações.
#
# Para os desafios de busca por PRODUTO, construiremos uma visão com
# uma entrada por código. Isto NÃO substitui o dataset transacional.

produtos_por_codigo = {}
for codigo, descricao in zip(df["StockCode"], df["Description"]):
    if pd.notna(codigo) and pd.notna(descricao):
        produtos_por_codigo[str(codigo)] = str(descricao)

registros = list(produtos_por_codigo.items())

print("Linhas transacionais:", len(df))
print("Produtos distintos na visão código → descrição:", len(registros))
print("Exemplo:", registros[:5])


## Atividade 1 — escolha antes da medida

**Pergunta aos estudantes:** qual representação escolheriam para cada operação e por quê?

Não corrija imediatamente. O objetivo é criar uma hipótese que será revisitada na Atividade 9.

**O que observar:** é provável que apareça `dict` para quase tudo depois da S05. Use isso como hipótese, não como erro.


In [ ]:
operacoes = [
    "código exato",
    "intervalo",
    "inserção",
    "grupos de clientes",
    "processamento em massa"
]
for operacao in operacoes:
    print(f"{operacao:24s} → peça uma escolha e uma justificativa")


**Discussão possível:** `dict` é uma boa hipótese para busca exata, mas ainda não sabemos se responde naturalmente a intervalos, álgebra de conjuntos ou processamento vetorizado. Não antecipe a conclusão.


## Atividade 2 — os mesmos pares, três formas de buscar

**Antes do código, pergunte:**
- Qual pode percorrer quase todos os elementos?
- Por que a binária exige ordenação?
- O que muda quando a chave está ausente?

**Solução:**


In [ ]:
def busca_sequencial(registros, alvo):
    for codigo, descricao in registros:
        if codigo == alvo:
            return descricao
    return None


def busca_binaria(registros_ordenados, alvo):
    inicio = 0
    fim = len(registros_ordenados) - 1

    while inicio <= fim:
        meio = (inicio + fim) // 2
        codigo, descricao = registros_ordenados[meio]

        if codigo == alvo:
            return descricao
        elif codigo < alvo:
            inicio = meio + 1
        else:
            fim = meio - 1

    return None


ordenada = sorted(registros)
indice = dict(registros)

alvo_presente = registros[-1][0]
alvo_ausente = "CODIGO_AUSENTE"

for alvo in [alvo_presente, alvo_ausente]:
    print("\nAlvo:", alvo)
    print("Sequencial:", busca_sequencial(registros, alvo))
    print("Binária:   ", busca_binaria(ordenada, alvo))
    print("Dict:      ", indice.get(alvo))


**Resultado esperado:** as três estratégias respondem à mesma pergunta.

**O que observar:** sequencial O(n); binária O(log n) depois da ordenação; `dict` O(1) médio depois da construção.

**Erro comum:** dizer que `dict` “não tem custo”. Ele teve custo de construção e ocupa uma representação adicional.

**Transição para o Mestre:** “Uma consulta não conta a história inteira.”


## Atividade 3 — preparação + muitas consultas

**Pergunta:** quando o custo inicial de organizar/indexar pode se pagar?

Peça previsão para 1, 100 e 10.000 consultas antes de executar.


In [ ]:
import timeit
import random
import statistics

def medir_buscas(registros,
                 tamanhos=(500, 2000, 4000),
                 quantidades=(1, 10, 100, 1000, 10000)):
    saida = []
    aleatorio = random.Random(5310)

    for n in tamanhos:
        base = registros[:min(n, len(registros))]
        chaves = [codigo for codigo, _ in base]
        if not chaves:
            continue

        consultas = [
            chaves[aleatorio.randrange(len(chaves))] if i % 2
            else "CODIGO_AUSENTE"
            for i in range(max(quantidades))
        ]

        # Estruturas usadas nas consultas.
        ordenada = sorted(base)
        indice = dict(base)

        preparos = {
            "sequencial": lambda: list(base),
            "binaria": lambda: sorted(base),
            "dict": lambda: dict(base),
        }

        funcoes = {
            "sequencial": lambda q: busca_sequencial(base, q),
            "binaria": lambda q: busca_binaria(ordenada, q),
            "dict": lambda q: indice.get(q),
        }

        for nome in funcoes:
            tprep = statistics.median(
                timeit.repeat(preparos[nome], number=1, repeat=3)
            )

            for k in quantidades:
                lote = consultas[:k]
                tconsulta = statistics.median(
                    timeit.repeat(
                        lambda: [funcoes[nome](q) for q in lote],
                        number=1,
                        repeat=3
                    )
                )

                saida.append(
                    (len(base), k, nome,
                     tprep, tconsulta, tprep + tconsulta)
                )

    return pd.DataFrame(
        saida,
        columns=["n", "consultas", "estrutura",
                 "preparo_s", "consultas_s", "total_s"]
    )


resultado_buscas = medir_buscas(registros)
display(
    resultado_buscas[
        resultado_buscas["consultas"].isin([1, 100, 10000])
    ].round(6)
)


**O que observar:** não prometa um ponto de cruzamento universal. Hardware, implementação, tamanho real da visão deduplicada e distribuição das consultas afetam os tempos.

**Erro comum:** comparar somente `indice.get()` com a busca sequencial e esquecer o custo de `dict(base)`.

**Discussão:** Big-O descreve crescimento; benchmark mostra constantes e custos concretos naquele ambiente.


## Atividade 4 — mudamos a pergunta: intervalo

**Pergunta:** por que uma estrutura excelente para chave exata pode não oferecer naturalmente uma consulta por faixa?

**Solução:**


In [ ]:
def busca_intervalo_linear(registros, inferior, superior):
    resultado = []
    for codigo, descricao in registros:
        if inferior <= codigo <= superior:
            resultado.append((codigo, descricao))
    return resultado


def limite_inferior(registros_ordenados, chave):
    inicio = 0
    fim = len(registros_ordenados)

    while inicio < fim:
        meio = (inicio + fim) // 2
        if registros_ordenados[meio][0] < chave:
            inicio = meio + 1
        else:
            fim = meio

    return inicio


def busca_intervalo_ordenada(registros_ordenados, inferior, superior):
    inicio = limite_inferior(registros_ordenados, inferior)
    resultado = []

    for posicao in range(inicio, len(registros_ordenados)):
        codigo, descricao = registros_ordenados[posicao]

        if codigo > superior:
            break

        resultado.append((codigo, descricao))

    return resultado


inferior, superior = "22000", "22999"

linear = busca_intervalo_linear(registros, inferior, superior)
ordenado = busca_intervalo_ordenada(ordenada, inferior, superior)
via_dict = [
    (codigo, descricao)
    for codigo, descricao in indice.items()
    if inferior <= codigo <= superior
]

assert set(linear) == set(ordenado) == set(via_dict)

print("Quantidade:", len(linear))
print("Primeiros da ordenada:", ordenado[:5])


**Resultado esperado:** lista não ordenada e `dict` precisam examinar as chaves para descobrir quais pertencem à faixa. A ordenada encontra o início por busca e percorre apenas a região relevante.

**Formalização didática:** ordenada ≈ O(log n + k); varreduras ≈ O(n), onde `k` é o número de resultados.

**Cuidado:** a faixa é lexicográfica porque `StockCode` está sendo tratado como texto.

**Transição:** “O `dict` não perdeu; mudamos a operação.”


## Exploração intermediária — multiplicidade e composição de estruturas

**Objetivo:** recuperar uma propriedade que a deduplicação por `StockCode` escondeu: no dataset transacional, repetição pode ser informação.

Use `InvoiceNo`: uma venda pode conter várias linhas/produtos.

Pergunte antes do código:

- eliminar `InvoiceNo` repetidos preservaria a venda?
- o que acontece com `d[invoice] = produto` quando repetimos a chave?
- precisamos abandonar `dict` ou mudar o valor associado à chave?

A chegada desejada é:

> **uma chave pode apontar para uma estrutura que contém vários valores.**


In [ ]:
contagem_faturas = df["InvoiceNo"].value_counts()
invoice_exemplo = contagem_faturas.index[0]

linhas_venda = df.loc[
    df["InvoiceNo"] == invoice_exemplo,
    ["InvoiceNo", "StockCode", "Description", "Quantity"]
]

print("InvoiceNo:", invoice_exemplo)
print("Quantidade de linhas:", len(linhas_venda))
display(linhas_venda.head(10))

vendas = {}

for linha in df[["InvoiceNo", "StockCode", "Quantity"]].itertuples(index=False):
    invoice = linha.InvoiceNo
    produto = str(linha.StockCode)
    quantidade = linha.Quantity

    if invoice not in vendas:
        vendas[invoice] = []

    vendas[invoice].append((produto, quantidade))

print(vendas[invoice_exemplo][:10])


**Discussão esperada:**

- `list` preserva ocorrências e ordem de inserção;
- `set` deliberadamente representa valores únicos e, portanto, não preserva multiplicidade;
- `dict[invoice] = produto` sobrescreve o valor anterior para a mesma chave;
- `dict[invoice] = list(...)` representa naturalmente uma relação 1:N;
- também seria possível representar frequências, por exemplo `produto → contagem`, se essa fosse a pergunta.

**Não diga:** “`dict` não permite vários valores”.

Diga:

> uma chave de `dict` está associada a um valor; esse valor pode ser outra estrutura de dados.


## Atividade 5 — problema genuinamente conjuntista

Antes da execução, explicite a notação:

- `A & B`: interseção;
- `A | B`: união;
- `A - B`: diferença;
- `A ^ B`: diferença simétrica = `(A - B) | (B - A)`.

Para a turma, `^` é o menos intuitivo: enfatize “está em exatamente um dos grupos”.

**Pergunta:** por que agora `set` é mais do que “outro jeito de fazer `in`”?

Peça primeiro o significado de `&`, `|`, `-` e `^` no contexto dos clientes.


In [ ]:
validos = df[df["CustomerID"].notna()]

A = set(
    validos.loc[
        validos["Country"] == "United Kingdom",
        "CustomerID"
    ].astype(int)
)

B = set(
    validos.loc[
        validos["Quantity"] >= 10,
        "CustomerID"
    ].astype(int)
)


def intersecao_didatica(A, B):
    resultado = set()
    # Percorrer o menor é uma boa decisão didática/algorítmica.
    menor, maior = (A, B) if len(A) <= len(B) else (B, A)
    for x in menor:
        if x in maior:
            resultado.add(x)
    return resultado


def diferenca_didatica(A, B):
    resultado = set()
    for x in A:
        if x not in B:
            resultado.add(x)
    return resultado


def uniao_didatica(A, B):
    resultado = set(A)
    for x in B:
        resultado.add(x)
    return resultado


assert intersecao_didatica(A, B) == A & B
assert diferenca_didatica(A, B) == A - B
assert uniao_didatica(A, B) == A | B

print("|A|:", len(A))
print("|B|:", len(B))
print("|A & B|:", len(A & B))
print("|A | B|:", len(A | B))
print("|A - B|:", len(A - B))
print("|A ^ B|:", len(A ^ B))

cliente = next(iter(A))
print("Exemplo de pertencimento:", cliente, cliente in A)


**O que observar:** `set` preserva unicidade durante a inserção; não “remove duplicados depois”.

**`x in A`:** hash → posição candidata → comparação → eventual probing em caso de colisão.

**`A & B`:** envolve vários testes de pertencimento. Uma linha de Python não implica O(1).

**Diferença:** `A - B` tem direção.

**Erro comum:** afirmar que `set` é sempre preferível a `dict` para pertencimento. Se precisamos de `chave → valor`, `dict` já resolve pertencimento e preserva a informação associada.


## Atividade 6 — `append`, capacidade e memória

**Antes de executar:** desenhe no quadro tamanho lógico 4 e capacidade conceitual 7. Simule um `append` com espaço e outro sem espaço.

Pergunte explicitamente: “Python dobra sempre? Remover metade devolve metade da memória?”


In [ ]:
import sys
import matplotlib.pyplot as plt

lista_memoria = []
crescimento = [(0, sys.getsizeof(lista_memoria))]

for i in range(5000):
    antes = sys.getsizeof(lista_memoria)
    lista_memoria.append(i)
    depois = sys.getsizeof(lista_memoria)

    if depois != antes:
        crescimento.append((len(lista_memoria), depois))


remocao = [(len(lista_memoria), sys.getsizeof(lista_memoria))]

for _ in range(5000):
    antes = sys.getsizeof(lista_memoria)
    lista_memoria.pop()
    depois = sys.getsizeof(lista_memoria)

    if depois != antes:
        remocao.append((len(lista_memoria), depois))


print("Primeiras mudanças no crescimento:")
print(crescimento[:10])

print("\nÚltimas mudanças na remoção:")
print(remocao[-10:])

xc, yc = zip(*crescimento)
xr, yr = zip(*remocao)

plt.figure(figsize=(9, 5))
plt.plot(xc, yc, marker=".", label="crescimento")
plt.plot(xr, yr, marker=".", label="remoção")
plt.xlabel("len(lista)")
plt.ylabel("sys.getsizeof(lista) [bytes]")
plt.title("Degraus de alocação observáveis em uma list do CPython")
plt.legend()
plt.grid(alpha=0.25)
plt.show()


**Resultado esperado:** degraus, não crescimento byte a byte.

**Explicação:** `list` é, de forma simplificada, um array dinâmico contíguo de referências. Quando precisa crescer, o CPython usa uma política de over-allocation; não ensine “sempre dobra”.

**O(1) amortizado:** a maioria dos `append`s usa capacidade já disponível; ocasionalmente ocorre redimensionamento mais caro.

**Remoção:** a estrutura não precisa encolher a cada `pop`. A política evita oscilações caras.

**Cuidado:** `sys.getsizeof()` mede a estrutura da lista, não recursivamente os objetos apontados. A capacidade exata não é exposta diretamente por essa função.


## Atividade 7 — inserção e manutenção

**Sentido de `bisect.insort`:** manter uma `list` ordenada enquanto novos elementos chegam.

Separe no quadro:

`localizar posição O(log n)` + `abrir espaço/deslocar O(n)` → **inserção O(n)**.

O valor didático de `bisect` aqui não é ensinar uma nova biblioteca, mas mostrar que **otimizar uma etapa não muda necessariamente o custo dominante da operação completa**.

**Pergunta:** qual propriedade estamos pagando para preservar?

**Solução:**


In [ ]:
import bisect
import timeit

amostra = registros[:min(3000, len(registros))]
novos = [
    (f"Z{i:06d}", f"Produto novo {i}")
    for i in range(200)
]


def inserir_append():
    lista = list(amostra)
    for item in novos:
        lista.append(item)
    return lista


def inserir_ordenado():
    lista = sorted(amostra)
    for item in novos:
        bisect.insort(lista, item)
    return lista


def inserir_dict():
    dicionario = dict(amostra)
    for codigo, descricao in novos:
        dicionario[codigo] = descricao
    return dicionario


def inserir_set():
    conjunto = {codigo for codigo, _ in amostra}
    for codigo, _ in novos:
        conjunto.add(codigo)
    return conjunto


for funcao in [
    inserir_append,
    inserir_ordenado,
    inserir_dict,
    inserir_set
]:
    tempo = min(timeit.repeat(funcao, number=1, repeat=3))
    print(f"{funcao.__name__:18s} {tempo*1000:.3f} ms")


**O que observar:** `bisect` encontra posição em O(log n), mas inserir em um array dinâmico desloca referências, portanto a inserção continua O(n).

`append` é O(1) amortizado; `dict`/`set` O(1) médio para inserção sob condições usuais.

**Cuidado pedagógico:** as quatro operações não preservam a mesma semântica. O `set` guarda apenas códigos; `append` não preserva ordenação.

**Transição:** estrutura eficiente para consultar pode impor custo para manter sua propriedade.


## Atividade 8 — processamento em massa

**Objetivo didático:** usar NumPy como contraponto experimental, não como nova estrutura a ser estudada internamente.

Problema: calcular `Quantity × UnitPrice` para muitas linhas.

Compare:

- laço explícito em Python;
- operação NumPy sobre arrays.

Pergunta antes do cronômetro:

> Se as duas soluções precisam processar `n` valores e são O(n), elas precisam ter o mesmo tempo concreto?

Não entrar em `dtype`, layout de memória, strides, views ou detalhes internos do `ndarray`.


In [ ]:
import timeit

precos = df["UnitPrice"].fillna(0).astype(float).tolist()
quantidades = df["Quantity"].fillna(0).astype(float).tolist()


def total_python(p, q):
    resultado = []
    for i in range(len(p)):
        resultado.append(p[i] * q[i])
    return sum(resultado)


def total_numpy(p, q):
    return float(np.sum(p * q))


for n in [1000, 10000, 100000, min(len(precos), 500000)]:
    p = precos[:n]
    q = quantidades[:n]

    ap = np.asarray(p, dtype=np.float64)
    aq = np.asarray(q, dtype=np.float64)

    valor_python = total_python(p, q)
    valor_numpy = total_numpy(ap, aq)

    assert np.isclose(
        valor_python, valor_numpy,
        rtol=1e-9, atol=1e-5
    )

    t_python = min(
        timeit.repeat(
            lambda: total_python(p, q),
            number=1, repeat=3
        )
    )

    t_numpy = min(
        timeit.repeat(
            lambda: total_numpy(ap, aq),
            number=1, repeat=3
        )
    )

    print(
        f"{n:7d} | "
        f"Python {t_python:.6f}s | "
        f"NumPy {t_numpy:.6f}s | "
        f"speedup ≈ {t_python/t_numpy:.1f}x"
    )

print("\nConversão list → ndarray é custo de preparação e ficou fora dos tempos acima.")


**Resultado esperado:** ambas são O(n), mas os tempos concretos podem diferir bastante.

A explicação suficiente nesta semana é:

> NumPy oferece implementações otimizadas para operações sobre arrays, evitando que escrevamos e executemos explicitamente o laço elemento a elemento em Python.

Pontos de discussão:

- Big-O descreve crescimento, não segundos;
- conversão com `np.asarray(...)` também tem custo;
- se o array será reutilizado muitas vezes, a preparação pode ser amortizada;
- não atribuir a diferença a “NumPy ser O(1)”;
- não aprofundar a implementação interna do `ndarray`.


## Atividade 9 — decisão revista

Peça que comparem explicitamente com a Atividade 1. Uma mudança de resposta é evidência de aprendizagem, não erro inicial.

**Síntese possível, não resposta única:**

- consulta exata repetida → `dict` é forte candidato;
- intervalo repetido → representação ordenada pode justificar seu custo de manutenção;
- grupos/unicidade/interseção → `set`;
- inserção → depende da propriedade que precisa ser preservada;
- lote numérico → `ndarray` quando a representação homogênea é adequada.

**Pergunta final:** se mantivermos simultaneamente um `dict` e uma estrutura ordenada, o que acontece quando um produto é inserido, removido ou alterado?

**Resposta esperada:** surge custo de sincronização/consistência entre representações.

A conclusão da semana é: **a estrutura vem depois da pergunta e do padrão de operações.**
